# Model Karşılaştırması (Baseline vs BERT)

Bu notebook'ta klasik Makine Öğrenmesi (Baseline) modelleri ile önceden eğitilmiş Derin Öğrenme (BERT) modelinin sonuçlarını karşılaştırıyoruz. Her iki yaklaşıma ait sonuç dosyalarını birleştirerek performans (Accuracy, F1 vs.) ve hız (Eğitim süresi) dengesini analiz edeceğiz.

## Bölüm 1 — Verileri Birleştir
- `baseline_results.csv` ve `bert_results.json` dosyalarını tek bir veri çerçevesinde birleştireceğiz.

In [1]:
import os
import json
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from IPython.display import display, Markdown

import warnings
warnings.filterwarnings('ignore')

try:
    # 1. baseline okuma
    baseline_df = pd.read_csv('data/processed/baseline_results.csv')
    
    # Eğitim Süresi sütunu isim kontrolü
    time_col = 'Eğitim Süresi (s)' if 'Eğitim Süresi (s)' in baseline_df.columns else 'Train Time (s)'
    
    # 2. bert okuma
    with open('data/processed/bert_results.json', 'r', encoding='utf-8') as f:
        bert_dict = json.load(f)
        
    bert_df = pd.DataFrame([bert_dict])
    
    # BERT için eğitim süresi 'N/A' ancak grafikler için önce sayısal 0 verelim, 
    # formatlanmış halinde 'N/A' kullanacağız.
    bert_df[time_col] = 0.0
    
    baseline_df['Süre Gösterimi'] = baseline_df[time_col].round(2).astype(str)
    bert_df['Süre Gösterimi'] = 'N/A (hazır model)'

    df_all = pd.concat([baseline_df, bert_df], ignore_index=True)
    
    print("\u2705 Veriler başarıyla birleştirildi.\n")
    print("--- Birleşik Model Sonuçları ---")
    display(df_all)
    
except Exception as e:
    print(f"Hata oluştu: {e}. Lütfen NB3 ve NB4'ü çalıştırıp dosyaların oluştuğundan emin olun.")

✅ Veriler başarıyla birleştirildi.

--- Birleşik Model Sonuçları ---


,Model,Accuracy,Precision,Recall,F1,Eğitim Süresi (s),Süre Gösterimi
0,LightGBM,0.687777,0.653056,0.687777,0.656232,21.995875,22.0
1,LinearSVC,0.644776,0.644340,0.644776,0.644420,7.552104,7.55
2,LogisticRegression,0.627995,0.644720,0.627995,0.635538,24.261291,24.26
3,MultinomialNB,0.692537,0.655607,0.692537,0.633872,0.016030,0.02
4,BERT,0.711000,0.764577,0.711000,0.695589,0.000000,N/A (hazır model)


## Bölüm 2 — Karşılaştırma Grafikleri
Model metriklerini çubuk (bar), radar ve serpilme (scatter) grafikleriyle Plotly üzerinden interaktif olarak görselleştireceğiz.

In [2]:
# Grafik 1: Metrik Karşılaştırması (Grouped Bar Chart)
metrics = ['Accuracy', 'Precision', 'Recall', 'F1']

fig1 = go.Figure()
for metric in metrics:
    fig1.add_trace(go.Bar(
        x=df_all['Model'],
        y=df_all[metric],
        name=metric,
        text=df_all[metric].round(3),
        textposition='auto'
    ))

fig1.update_layout(
    title="Model Performans Karşılaştırması",
    xaxis_title="Modeller",
    yaxis_title="Skorlar",
    barmode='group',
    template="plotly_white"
)
fig1.show()

In [3]:
# Grafik 2: Radar Chart
fig2 = go.Figure()

for i, row in df_all.iterrows():
    fig2.add_trace(go.Scatterpolar(
        r=[row['Accuracy'], row['Precision'], row['Recall'], row['F1'], row['Accuracy']],
        theta=['Accuracy', 'Precision', 'Recall', 'F1', 'Accuracy'],
        fill='toself',
        name=row['Model'],
        opacity=0.6
    ))

fig2.update_layout(
    polar=dict(
        radialaxis=dict(visible=True, range=[0, 1])
    ),
    showlegend=True,
    title="Model Profil Karşılaştırması",
    template="plotly_white"
)
fig2.show()

In [4]:
# Grafik 3: Hız vs Performans (Scatter Plot)
df_all['Kategori'] = df_all['Model'].apply(lambda x: 'Hazır Model (BERT)' if 'BERT' in x else 'Eğitilen Baseline')

fig3 = px.scatter(
    df_all, 
    x=time_col, 
    y='F1', 
    color='Kategori',
    symbol='Kategori',
    text='Model',
    title="Hız — Performans Dengesi",
    size_max=15
)
fig3.update_traces(textposition='top center', marker=dict(size=12))
fig3.update_layout(
    xaxis_title="Eğitim Süresi (saniye)", 
    yaxis_title="F1 Skoru",
    template="plotly_white"
)
fig3.show()

## Bölüm 3 — Sonuç Tablosu
Tüm modelleri F1 skoruna göre sıralayıp, en iyi ve en kötü olanları tespit ediyoruz.

In [5]:
df_sorted = df_all.sort_values(by='F1', ascending=False).reset_index(drop=True)

def highlight_max_min(s):
    if s.name == 'F1':
        is_max = s == s.max()
        is_min = s == s.min()
        return ['background-color: rgba(46, 204, 113, 0.4); font-weight: bold' if v 
                else 'background-color: rgba(231, 76, 60, 0.4)' if m else '' for v, m in zip(is_max, is_min)]
    return [''] * len(s)

print("--- F1 Skoruna Göre Sıralı Modeller ---")
display(df_sorted[['Model', 'Accuracy', 'Precision', 'Recall', 'F1', 'Süre Gösterimi']]
        .style.apply(highlight_max_min).format({'Accuracy': "{:.4f}", 'Precision': "{:.4f}", 'Recall': "{:.4f}", 'F1': "{:.4f}"}))

best_baseline_f1 = df_sorted[df_sorted['Model'] != 'BERT']['F1'].max()
bert_f1 = df_sorted[df_sorted['Model'] == 'BERT']['F1'].values[0] if 'BERT' in df_sorted['Model'].values else 0

fark = bert_f1 - best_baseline_f1
durum = "iyi" if fark > 0 else "kötü"
yuzde_fark = abs(fark) * 100

print(f"\nBERT, en iyi baseline modelden F1 skorunda {abs(fark):.4f} puan daha {durum} performans gösterdi.")

--- F1 Skoruna Göre Sıralı Modeller ---


,Model,Accuracy,Precision,Recall,F1,Süre Gösterimi
0,BERT,0.7110,0.7646,0.7110,0.6956,N/A (hazır model)
1,LightGBM,0.6878,0.6531,0.6878,0.6562,22.0
2,LinearSVC,0.6448,0.6443,0.6448,0.6444,7.55
3,LogisticRegression,0.6280,0.6447,0.6280,0.6355,24.26
4,MultinomialNB,0.6925,0.6556,0.6925,0.6339,0.02



BERT, en iyi baseline modelden F1 skorunda 0.0394 puan daha iyi performans gösterdi.


## Bölüm 4 — Detaylı Yorum
Aşağıdaki hücre, analiz sonuçlarına bağlı olarak otomatik değerlendirme raporu oluşturur.

In [6]:
best_model_name = df_sorted.iloc[0]['Model']
worst_model_name = df_sorted.iloc[-1]['Model']
hiz_kritik_model = df_sorted[df_sorted['Model'] != 'BERT'].sort_values(by=[time_col]).iloc[0]['Model']

if yuzde_fark < 1:
    fark_yorumu = "anlamlı bir fark yok, baseline yeterli."
elif yuzde_fark <= 5:
    fark_yorumu = "orta düzey iyileşme, kullanım senaryosuna göre değerlendirilmeli."
else:
    fark_yorumu = "anlamlı iyileşme, BERT kesinlikle tercih edilmeli."

markdown_text = f"""
**1. Genel Sıralama:** Tabloya ve grafiklere bakıldığında, en yüksek F1 skoruna sahip model **{best_model_name}** olurken, en düşük performansı **{worst_model_name}** göstermiştir.

**2. BERT vs Baseline:** BERT ile en iyi klasik model arasındaki F1 farkı **%{(yuzde_fark):.2f}**. 
   - **Değerlendirme:** *{fark_yorumu}*

**3. Hız-Doğruluk Dengesi:** Eğer sistemin çok hızlı çalışması ve hemen cevap vermesi (veya eğitilmesi) kritikse, süresi en kısa olan **{hiz_kritik_model}** veya *Logistic Regression* gibi bir model önerilebilir. BERT, doğruluk getirse de kaynak (GPU/RAM) tüketir.

**4. Production (Canlı Ortam) Önerisi:** 
   - Streamlit arayüzü normal bilgisayarlarda (CPU) çalışacaksa ve tahminlerin gecikmesi (latency) kullanıcı deneyimini bozacaksa, en iyi Baseline modeli kullanılmalıdır.
   - Ancak sunucuda yeterli kaynak varsa ve müşteri şikayetlerinin (negative) tespiti hayati önem taşıyorsa, en iyi performansa sahip olan **{best_model_name}** modeli canlıya (production) alınmalıdır.
"""
display(Markdown(markdown_text))


**1. Genel Sıralama:** Tabloya ve grafiklere bakıldığında, en yüksek F1 skoruna sahip model **BERT** olurken, en düşük performansı **MultinomialNB** göstermiştir.

**2. BERT vs Baseline:** BERT ile en iyi klasik model arasındaki F1 farkı **%3.94**. 
   - **Değerlendirme:** *orta düzey iyileşme, kullanım senaryosuna göre değerlendirilmeli.*

**3. Hız-Doğruluk Dengesi:** Eğer sistemin çok hızlı çalışması ve hemen cevap vermesi (veya eğitilmesi) kritikse, süresi en kısa olan **MultinomialNB** veya *Logistic Regression* gibi bir model önerilebilir. BERT, doğruluk getirse de kaynak (GPU/RAM) tüketir.

**4. Production (Canlı Ortam) Önerisi:** 
   - Streamlit arayüzü normal bilgisayarlarda (CPU) çalışacaksa ve tahminlerin gecikmesi (latency) kullanıcı deneyimini bozacaksa, en iyi Baseline modeli kullanılmalıdır.
   - Ancak sunucuda yeterli kaynak varsa ve müşteri şikayetlerinin (negative) tespiti hayati önem taşıyorsa, en iyi performansa sahip olan **BERT** modeli canlıya (production) alınmalıdır.


## Bölüm 5 — Kaydetme
Analizler sonucunda elde edilen nihai tabloyu CSV olarak kaydediyoruz.

In [7]:
output_dir = 'data/processed'
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

df_final = df_sorted[['Model', 'Accuracy', 'Precision', 'Recall', 'F1', 'Süre Gösterimi']].copy()
df_final.rename(columns={'Süre Gösterimi': 'Eğitim Süresi (s)'}, inplace=True)

save_path = os.path.join(output_dir, 'model_comparison.csv')
df_final.to_csv(save_path, index=False)

print(f"\u2705 Karşılaştırma tablosu '{save_path}' yoluna başarıyla kaydedildi.")

✅ Karşılaştırma tablosu 'data/processed\model_comparison.csv' yoluna başarıyla kaydedildi.
